In [1]:
import re
import random
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
import scipy.stats as stats
from scipy.stats import norm
from pathlib import Path
import optuna
import os

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [3]:
os.environ["OMP_NUM_THREADS"]  = "1"
os.environ["MKL_NUM_THREADS"]  = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"]  = "1"
torch.set_num_threads(1)        # PyTorch intra-op
torch.set_num_interop_threads(1)  # PyTorch inter-op

In [4]:
def split_samples_random(base_folder, train_ratio=0.8, val_ratio=0.1, seed=42):
    base = Path(base_folder)
    # Gather all sample IDs
    sample_ids = sorted([
        int(d.name.split("_")[-1])
        for d in base.iterdir()
        if d.is_dir() and d.name.startswith("sample_")
    ])
    # Shuffle and split
    random.seed(seed)
    random.shuffle(sample_ids)
    n = len(sample_ids)
    n_train = int(train_ratio * n)
    n_val = int(val_ratio * n)
    train_ids = sample_ids[:n_train]
    val_ids = sample_ids[n_train:n_train + n_val]
    test_ids = sample_ids[n_train + n_val:]
    # Collect CSVs for each split
    def gather(ids):
        files = []
        for sid in ids:
            for csv in (base / f"sample_{sid:02d}").glob("*.csv"):
                files.append(csv)
        return files

    return gather(train_ids), gather(val_ids), gather(test_ids)

In [5]:
base_folder = r"/scratch/bkmanu/NIH/spike-1.6.0rc2-linux64/Simulated_data/Sim_no_centering_new_funct"
train_files, val_files, test_files = split_samples_random(base_folder)

In [ ]:
train_files

In [7]:
target_path = r"/scratch/bkmanu/NIH/spike-1.6.0rc2-linux64/Simulated_data/Sim_no_centering_new_funct/constants.csv"

In [8]:
def load_coefs(coef_csv_path):

    df = pd.read_csv(coef_csv_path).set_index('sample_id')
    # Drop columns that are constant across all rows
    non_constant_df = df.loc[:, (df != df.iloc[0]).any()]
    coef_names = list(non_constant_df.columns)
    coef_map = {
        int(sample_id): row.values.astype(float)
        for sample_id, row in non_constant_df.iterrows()
    }
    return coef_map, coef_names

In [ ]:
coef_map, coef_names = load_coefs(coef_csv_path=target_path)
coef_names

In [10]:
def load_data(file_list, constants_csv="constants.csv"):
    # Read the map and the names
    coef_map, coef_names = load_coefs(constants_csv)
    pattern = re.compile(r"sample_(\d+)_stepwise.csv$")

    X_list, y_list = [], []
    for f in file_list:
        f = Path(f)
        m = pattern.search(f.name)
        if not m:
            continue
        sid = int(m.group(1))
        if sid not in coef_map:
            continue

        df = pd.read_csv(f).drop(columns=['Time'], errors='ignore')
        if df.shape[0] != 366:
            continue

        X_list.append(df.values.astype(float))
        y_list.append(coef_map[sid])

    if not X_list:
        return np.empty((0,0,0)), np.empty((0, len(coef_names))), coef_names

    return np.stack(X_list, axis=0), np.stack(y_list, axis=0), coef_names

In [11]:
X_train, y_train, const_vals = load_data(train_files, constants_csv=target_path)
X_val,   y_val,   const_vals = load_data(val_files,   constants_csv=target_path)
X_test,  y_test,  _          = load_data(test_files,  constants_csv=target_path)

if y_train.ndim == 1:
    y_train = y_train.reshape(1, -1)
if y_test.ndim == 1:
    y_test = y_test.reshape(1, -1)
if y_val.ndim == 1:
    y_val = y_val.reshape(1, -1)

In [ ]:
X_train.shape
# y_val.shape
# y_test.shape

In [13]:
# Normalize data
def standardize_features(X_train, X_test, X_val):
    N_train, T, F = X_train.shape
    N_test = X_test.shape[0]
    N_val = X_val.shape[0]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train.reshape(-1, F)).reshape(N_train, T, F)
    X_test_scaled = scaler.transform(X_test.reshape(-1, F)).reshape(N_test, T, F)
    X_val_scaled = scaler.transform(X_val.reshape(-1, F)).reshape(N_val, T, F)

    return X_train_scaled, X_test_scaled, X_val_scaled, scaler

X_train, X_test, X_val, input_scaler = standardize_features(X_train, X_test, X_val)
y_train, y_test, y_val = y_train, y_test, y_val

In [14]:
class ResidualBlock(nn.Module):
    def __init__(self, channels, kernel_size=5, dropout=0.2):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=padding)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=padding)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = F.relu(out)
        out = self.dropout(out)

        out = self.conv2(out)
        out = self.dropout(out)

        # skip connection
        out += residual
        return F.relu(out)


class Conv1DRegressor(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_filters: int = 128,
        kernel_size: int = 5,
        num_blocks: int = 3,
        output_dim: int = 12,
        dropout: float = 0.2
    ):

        super().__init__()

        # Initial projection from input_dim -> num_filters
        self.input_proj = nn.Sequential(
            nn.Conv1d(input_dim, num_filters, kernel_size=1),
            nn.ReLU()
        )

        # Stacked residual blocks
        blocks = []
        for _ in range(num_blocks):
            blocks.append(ResidualBlock(num_filters, kernel_size, dropout))
        self.encoder = nn.Sequential(*blocks)

        # Pool to a single vector per sample
        self.global_pool = nn.AdaptiveAvgPool1d(1)

        # Head
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(num_filters, output_dim)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d) or isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
    
        # (batch, input_dim, seq_len)
        x = x.permute(0, 2, 1)

        # project & encode
        x = self.input_proj(x)
        x = self.encoder(x)

        # global avg pool -> (batch, num_filters, 1)
        x = self.global_pool(x)

        # -> (batch, num_filters)
        x = x.squeeze(-1)

        x = F.relu(x)
        x = self.dropout(x)

        out = self.fc(x)
        out = torch.sigmoid(out)
        return out

In [15]:
# Initialize model
batch_size = 8
# lr = 1e-4
# epochs = 50

In [16]:
class NoisyTensorDataset(Dataset):
    def __init__(self, X, y, noise_std=0.05, noise_type="gaussian"):
        self.X = X
        self.y = y
        self.s = noise_std
        self.kind = noise_type

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].astype(np.float32).copy()
        if self.s > 0:
            x += np.random.normal(0, self.s, size=x.shape)
        return torch.from_numpy(x), torch.from_numpy(self.y[idx].astype(np.float32))

In [17]:
train_ds = NoisyTensorDataset(
    X_train, y_train,
    noise_std=0.05,
    noise_type="gaussian"
)

val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                       torch.tensor(y_val, dtype=torch.float32))

test_ds = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                        torch.tensor(y_test, dtype=torch.float32))

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size)
test_loader = DataLoader(test_ds, batch_size=batch_size)

In [18]:
def train_model(
    model: nn.Module,
    train_loader: torch.utils.data.DataLoader,
    val_loader: torch.utils.data.DataLoader = None,
    epochs: int = 40,
    lr: float = 1e-4,
    device: torch.device = None,
    patience: int = 10         
):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    train_losses = []
    val_losses = []

    best_val_loss = float("inf")
    best_model_state = None
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        # Training
        model.train()
        total_train_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device).float()
            yb = yb.to(device).float()

            preds = model(xb)
            loss = loss_fn(preds, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation
        if val_loader is not None:
            model.eval()
            total_val_loss = 0.0
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb = xb.to(device).float()
                    yb = yb.to(device).float()
                    preds = model(xb)
                    total_val_loss += loss_fn(preds, yb).item()
            avg_val_loss = total_val_loss / len(val_loader)
            val_losses.append(avg_val_loss)
            print(f"Epoch {epoch}/{epochs} ▶ train loss: {avg_train_loss:.4f} | val loss: {avg_val_loss:.4f}")

            # Early Stopping Check
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_model_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch} (val loss did not improve for {patience} epochs)")
                    break

    # Restore best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    # Plot losses
    plt.figure(figsize=(6, 4))
    plt.plot(train_losses, label="Train")
    if val_loader is not None:
        plt.plot(val_losses, label="Val")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.title("Loss over epochs")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return train_losses, val_losses

In [ ]:
# Optuna objective and run study
def objective(trial):
    lr          = trial.suggest_float("lr", 1e-4, 3e-2, log=True)
    batch_size  = trial.suggest_categorical("batch_size", [8, 16, 32])
    dropout     = trial.suggest_float("dropout", 0.0, 0.5)

    # DataLoaders
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=False)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size*2, shuffle=False, num_workers=0, pin_memory=False)

    # Model
    model = Conv1DRegressor(
        input_dim=X_train.shape[2],
        output_dim=y_train.shape[1],
        dropout=dropout,
    )

    train_losses, val_losses = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=40,
        lr=lr,
        patience=10,
    )

    # Minimize best val loss
    return float(np.min(val_losses)) if len(val_losses) else float(np.min(train_losses))

# Run study
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, n_trials=10, n_jobs=6)

best_params = study.best_trial.params
best_params

In [20]:
import json
metadata = {
    "trial_number": study.best_trial.number,
    "best_value": study.best_value,
    "params": best_params
}

with open("best_params_with_meta.json", "w") as f:
    json.dump(metadata, f, indent=4)

In [19]:
def build_model_from_params(p):
    return Conv1DRegressor(
        input_dim=X_train.shape[2],
        output_dim=y_train.shape[1],
        dropout=p["dropout"],
    )

In [ ]:
# final model retraining with best params and save weights
final_model = build_model_from_params(best_params)

train_loader = DataLoader(train_ds, batch_size=best_params["batch_size"], shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=best_params["batch_size"]*2, shuffle=False)

_ = train_model(
    model=final_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=100,
    lr=best_params["lr"],
    patience=10,
)

torch.save(final_model.state_dict(), "best_model_new.pt")

In [24]:
def enable_dropout_only(module: nn.Module):
    for m in module.modules():
        if isinstance(m, (nn.Dropout, nn.Dropout1d, nn.Dropout2d, nn.Dropout3d,
                          nn.AlphaDropout, nn.FeatureAlphaDropout)):
            m.train()

@torch.no_grad()
def evaluate_model(
    model,
    loader,
    n_mc_samples: int = 1,
    device: torch.device = None,
    param_names=None,
    save_prefix: str = "predicted_parameters",
    do_plots: bool = False):
    
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    if param_names is None:
        param_names = [
            'lambda1_MH','lambda2_MH','gamma1_HM','gamma2_HM',
            'delta1_MH','delta2_MH','eta1_HM','eta2_HM',
            'p1_mortality','p2_mortality','iota1','iota2'
        ]

    # eval mode; re-enable only dropout layers for MC
    model.eval()
    if n_mc_samples > 1:
        enable_dropout_only(model)

    all_means, all_stds, all_trues = [], [], []

    for xb, yb in loader:
        xb = xb.to(device).float()
        yb = yb.to(device).float()

        if n_mc_samples == 1:
            pred = model(xb).detach().cpu().numpy()
            mean_pred = pred
            std_pred  = np.zeros_like(pred)
        else:
            preds = []
            for _ in range(n_mc_samples):
                out = model(xb)
                preds.append(out.detach().cpu().numpy())
            preds = np.stack(preds, axis=0)  # (T, B, D)
            mean_pred = preds.mean(axis=0)
            std_pred  = preds.std(axis=0, ddof=0)

        all_means.append(mean_pred)
        all_stds.append(std_pred)
        all_trues.append(yb.detach().cpu().numpy())

    all_means = np.concatenate(all_means, axis=0)
    all_stds  = np.concatenate(all_stds,  axis=0)
    all_trues = np.concatenate(all_trues, axis=0)

    # metrics (printed)
    rows = []
    for i, name in enumerate(param_names):
        true_i = all_trues[:, i]
        pred_i = all_means[:, i]
        std_i  = all_stds[:, i]
        mse  = float(np.mean((pred_i - true_i)**2))
        rmse = float(np.sqrt(mse))
        rows.append({
            'parameter': name,
            'Mean_True': float(true_i.mean()),
            'Mean_Pred': float(pred_i.mean()),
            'Bias':      float(pred_i.mean() - true_i.mean()),
            'MSE':       mse,
            'RMSE':      rmse,
            'Mean_STD':  float(std_i.mean()),
            'Coverage_1σ': float(np.mean(np.abs(pred_i - true_i) <= np.maximum(std_i, 1e-12))) if n_mc_samples > 1 else np.nan
        })
    df_metrics = pd.DataFrame(rows).set_index('parameter')
    print(df_metrics)

    # save wide CSV
    df = pd.DataFrame(all_means, columns=[f"pred_{n}" for n in param_names])
    for i, n in enumerate(param_names):
        df[f"true_{n}"] = all_trues[:, i]
        if n_mc_samples > 1:
            df[f"std_{n}"] = all_stds[:, i]

    csv_name = (
        f"{save_prefix}_with_uncertainty_T{n_mc_samples}.csv"
        if n_mc_samples > 1 else
        f"{save_prefix}.csv"
    )
    df.to_csv(csv_name, index=False)

    model.eval()
    return df_metrics, df

In [ ]:
best_model = Conv1DRegressor(
    input_dim=X_train.shape[2],
    num_filters=128,   
    kernel_size=5,     
    num_blocks=3,   
    output_dim=y_train.shape[1],   
    dropout=0.07664023472725412         
)
 
best_model.load_state_dict(torch.load("best_model_new.pt", map_location="cpu"))

best_model.eval()

In [ ]:
def mc_sweep(model, test_loader, n_list, device=None):
    results = {}
    for T in n_list:
        print(f"\n=== Evaluating with {T} stochastic passes ===")
        df_metrics, df_preds = evaluate_model(
            model,
            test_loader,
            n_mc_samples=T,
            device=device,
            save_prefix="predicted_parameters"
        )
        results[T] = {"metrics": df_metrics, "preds": df_preds}
    return results
    
mc_results = mc_sweep(best_model, test_loader, n_list=[10, 100, 250, 500])

In [ ]:
T_list = [10, 100, 250, 500]

val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, pin_memory=True)

for T in T_list:
    _ = evaluate_model(
        best_model,
        val_loader,
        n_mc_samples=T,
        save_prefix=f"predicted_parameters_with_uncertainty_val_T{T}",
        do_plots=False)

In [30]:
def fit_std_scales_from_val_csv(val_csv_path):
    df = pd.read_csv(val_csv_path)
    params = [c.replace("std_","") for c in df.columns if c.startswith("std_")]
    scales = {}
    for p in params:
        err2 = (df[f"pred_{p}"] - df[f"true_{p}"]).to_numpy()**2
        sig2 = (df[f"std_{p}"].to_numpy()**2)
        s2   = np.mean(err2 / sig2)         
        scales[p] = float(np.sqrt(max(s2, 0.0)))
    return scales

def apply_std_scales_to_csv(in_csv_path, out_csv_path, scales):
    df = pd.read_csv(in_csv_path).copy()
    for p, s in scales.items():
        col = f"std_{p}"
        if col in df.columns:
            df[col] = df[col] * s
    df.to_csv(out_csv_path, index=False)
    return df

for T in T_list:
    val_csv  = f"predicted_parameters_with_uncertainty_val_T{T}.csv"
    test_csv = f"predicted_parameters_with_uncertainty_T{T}.csv"
    scales   = fit_std_scales_from_val_csv(val_csv)
    _ = apply_std_scales_to_csv(
        in_csv_path=test_csv,
        out_csv_path=f"predicted_parameters_with_uncertainty_test_T{T}_CAL_tuned.csv",
        scales=scales)

In [32]:
def calibration_curve_from_csv(csv_path, param, levels=(0.5, 0.68, 0.9, 0.95)):
    df = pd.read_csv(csv_path)
    mu  = df[f"pred_{param}"].to_numpy()
    y   = df[f"true_{param}"].to_numpy()
    std = df[f"std_{param}"].to_numpy()
    lv   = np.asarray(levels, float)
    z_th = norm.ppf((1.0 + lv) / 2.0)
    empirical = [(np.abs(mu - y) <= z * std).mean() for z in z_th]
    return lv, np.array(empirical)

# Overlay different T values (Pre-recalibration and Post-recalibration)
def plot_param_calibration_over_T(
    param, T_list,
    test_csv_template      = "predicted_parameters_with_uncertainty_T{T}.csv",
    test_csv_cal_template  = "predicted_parameters_with_uncertainty_test_T{T}_CAL_tuned.csv",
    levels=(0.5,0.68,0.9,0.95),
    show_legend=True
):
    plt.figure(figsize=(6,4))
    lv = np.asarray(levels, float)
    plt.plot(lv,lv,'k--',label="Ideal")

    for T in T_list:
        unc_path = test_csv_template.format(T=T)
        cal_path = test_csv_cal_template.format(T=T)
        lv, emp_u = calibration_curve_from_csv(unc_path, param, levels)
        _,  emp_c = calibration_curve_from_csv(cal_path, param, levels)

        plt.plot(lv, emp_u, 'o--', label=f"T={T} Pre-recalibration")
        plt.plot(lv, emp_c, 'o-',  label=f"T={T} Post-recalibration")

    plt.xlabel("Nominal coverage"); plt.ylabel("Empirical coverage")
    plt.title(f"Calibration over T: {param}")
    plt.grid(True, linestyle="--", alpha=0.5)
    if show_legend:
        plt.legend(ncol=2)
    plt.tight_layout()
    plt.show()

In [ ]:
T_list = [10, 100, 250, 500]
params_to_show = ['lambda1_MH','lambda2_MH','gamma1_HM','gamma2_HM',
                  'delta1_MH','delta2_MH','eta1_HM','eta2_HM',
                  'p1_mortality','p2_mortality','iota1','iota2']

plot_param_calibration_over_T(params_to_show, T_list)